In [35]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [36]:
d = pd.read_parquet('ff_daily_garchdata_long.parquet')
r = d.loc[d.series == 'Mkt-RF', 'ret'].to_numpy()[-2500:]*100
r = torch.tensor(r, dtype = torch.float64)
r_train = r[:2000]

In [37]:
raw = torch.tensor([-1.0, 1.39, 3.81], dtype = torch.float64, requires_grad = True)

In [38]:
def unpack(raw):
    omega = torch.nn.functional.softplus(raw[0]) * 0.1
    w = torch.softmax(torch.concat([raw[1:], torch.zeros(1, dtype = raw.dtype)]), dim = 0)
    return omega, w[0], w[1]

In [39]:
#Negative log-likelihood function
def nll(raw, r):
    omega, alpha, beta = unpack(raw)
    h = r.var()
    total = torch.zeros((),dtype = r.dtype)
    for t in range(1, len(r)):
        h = omega + alpha * r[t-1]**2 + beta * h
        total = total + torch.log(h) + r[t]**2 / h
    return 0.5 * total / (len(r) - 1)

In [40]:
opt = torch.optim.Adam([raw], lr = 0.05)
for step in range(301):
    opt.zero_grad()
    loss = nll(raw,r)
    loss.backward()
    opt.step()
    if step % 50 == 0:
        print(f'Step {step} | Loss {round(loss.item(),5)} | Theta Space {[round(x.item(),4) for x in unpack(raw)]}')

Step 0 | Loss -0.0065 | Theta Space [0.03, 0.0876, 0.8916]
Step 50 | Loss -0.11745 | Theta Space [0.015, 0.1768, 0.7941]
Step 100 | Loss -0.11834 | Theta Space [0.0122, 0.166, 0.8053]
Step 150 | Loss -0.11842 | Theta Space [0.0114, 0.16, 0.8126]
Step 200 | Loss -0.11846 | Theta Space [0.0111, 0.1594, 0.8144]
Step 250 | Loss -0.11849 | Theta Space [0.0109, 0.1593, 0.8156]
Step 300 | Loss -0.11851 | Theta Space [0.0107, 0.1593, 0.8165]
